# 02 - Exploratory data analysis

**Goal:** understand the clean data: missing values, the two targets, time, what drives a repeat purchase and the spend, the categorical columns, the customers, and how test compares with train. Nothing is changed here.

- Uses the clean data saved by `01_data_cleaning.ipynb` (run it first).
- Each section ends with a **Key insight**: what we found, and what to do in preprocessing.
- The features suggested by these findings are built in `03_feature_engineering_selection.ipynb`.

In [41]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = "../data"
TARGET, SPEND = "repeat_purchase_90d", "spend_next_90d"
BLUE, ORANGE, GREY, RED = "#2f6690", "#d9822b", "#9aa5b1", "#c0392b"

# Same plot style as notebook 01
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(size=13), title=dict(font=dict(size=17)), height=450,
    margin=dict(t=80, l=60, r=30, b=50), hoverlabel=dict(bgcolor="white"),
    uniformtext=dict(minsize=11, mode="show")))
pio.templates.default = "plotly_white+eda"

# Clean data saved at the end of notebook 01 (run 01 first), plus the raw customer table
# The categorical labels are numbers stored as text ("7"): read them as text, not as numbers
CAT_COLS = ["payment_type", "salesperson", "commercial_zone", "transport_method", "warehouse", "document_series"]
df = pd.read_csv(f"{DATA_DIR}/processed/train_clean.csv", parse_dates=["purchase_date"], dtype={c: str for c in CAT_COLS})
test = pd.read_csv(f"{DATA_DIR}/processed/test_clean.csv", parse_dates=["purchase_date"], dtype={c: str for c in CAT_COLS})
clientes = pd.read_csv(f"{DATA_DIR}/clientes.csv")
print("train:", df.shape, "| test:", test.shape, "| clientes:", clientes.shape)

column_groups = {
    "categoricals": CAT_COLS,
    "hist_timing": ["days_since_previous_purchase", "customer_tenure_days", "prior_purchase_count",
                    "prior_mean_gap_days", "prior_std_gap_days"],
    "hist_value": ["prior_total_spend", "prior_mean_purchase_value", "prior_std_purchase_value",
                   "prior_min_purchase_value", "prior_max_purchase_value"],
}


def rate_by_bins(data, column, bins, labels=None, fig=None, row=None, col=None, title=None, xlabel=None):
    """Bar chart of the repeat-purchase rate per bin of `column`, with the row count on each bar.
    Draws into subplot (row, col) of `fig` when given, otherwise shows its own figure."""
    groups = pd.cut(data[column], bins=bins, labels=labels)
    summary = data.groupby(groups, observed=True)[TARGET].agg(["mean", "size"])
    own_figure = fig is None
    fig = go.Figure() if own_figure else fig
    pos = {} if own_figure else dict(row=row, col=col)
    overall = data[TARGET].mean()
    fig.add_bar(x=summary.index.astype(str), y=summary["mean"], marker_color=BLUE, showlegend=False,
                text=[f"n={n}" for n in summary["size"]], textposition="outside",
                hovertemplate="%{x}<br>repeat rate: %{y:.2f}<br>%{text}<extra></extra>", **pos)
    fig.add_hline(y=overall, line_dash="dash", line_color=ORANGE, **pos)
    fig.update_yaxes(range=[0, 1.1], title_text="repeat rate", **pos)
    fig.update_xaxes(title_text=xlabel or column, **pos)
    if own_figure:
        fig.update_layout(title=f"{title or f'Repeat rate by {column}'}<br><sup>dashed line = overall rate ({overall:.2f})</sup>")
        fig.show()
    return summary


def add_density(fig, values, color, name, row=1, col=1, showlegend=True):
    """Smoothed distribution (KDE) of `values` as a filled line in subplot (row, col)."""
    from scipy.stats import gaussian_kde
    values = values[np.isfinite(values)]
    grid = np.linspace(values.min(), values.max(), 200)
    r, g, b = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    fig.add_scatter(x=grid, y=gaussian_kde(values)(grid), mode="lines", fill="tozeroy", name=name,
                    line=dict(color=color, width=2), fillcolor=f"rgba({r},{g},{b},0.3)",
                    legendgroup=name, showlegend=showlegend, row=row, col=col)

train: (6469, 39) | test: (986, 37) | clientes: (1143, 13)


## 1. Missing values: structural or random?

In [42]:
first_purchase = df["prior_purchase_count"] == 0
one_prior = df["prior_purchase_count"] == 1
two_prior = df["prior_purchase_count"] == 2
# Each customer has at most one 1st and one 2nd purchase, so these also count customers
print("Customers in train:", df["customer_id"].nunique(),
      "| with their 1st purchase in train:", df.loc[first_purchase, "customer_id"].nunique(),
      "| with a 2nd purchase in train:", df.loc[one_prior, "customer_id"].nunique())

hist_cols = column_groups["hist_timing"] + column_groups["hist_value"] + ["spend_365d"]
# Which NaN are expected by definition?
needs_1_past = [c for c in hist_cols if c != "spend_365d"]                     # 1st purchase: no history (spend_365d should be 0, not NaN)
needs_2_past = ["prior_std_purchase_value", "prior_std_gap_days", "prior_mean_gap_days"]  # std and gaps need 2 past purchases
needs_3_past = ["prior_std_gap_days"]                                          # std of the gaps needs 2 gaps = 3 past purchases

miss = pd.DataFrame({"missing": df[hist_cols].isna().sum()})
miss["expected: 1st purchase"] = df.loc[first_purchase, hist_cols].isna().sum().where(miss.index.isin(needs_1_past), 0)
miss["expected: 2nd purchase"] = df.loc[one_prior, hist_cols].isna().sum().where(miss.index.isin(needs_2_past), 0)
miss["expected: 3rd purchase"] = df.loc[two_prior, hist_cols].isna().sum().where(miss.index.isin(needs_3_past), 0)
miss["unexplained"] = miss["missing"] - miss.iloc[:, 1:].sum(axis=1)
miss[miss["missing"] > 0]

Customers in train: 607 | with their 1st purchase in train: 603 | with a 2nd purchase in train: 445


,missing,expected: 1st purchase,expected: 2nd purchase,expected: 3rd purchase,unexplained
days_since_previous_purchase,796,603,0,0,193
prior_mean_gap_days,1244,603,445,0,196
prior_std_gap_days,1413,603,445,365,0
prior_mean_purchase_value,603,603,0,0,0
prior_std_purchase_value,1048,603,445,0,0
prior_min_purchase_value,603,603,0,0,0
prior_max_purchase_value,603,603,0,0,0
spend_365d,205,0,0,0,205


In [43]:
cat_cols = column_groups["categoricals"]
cat_missing = df[cat_cols].isna()
print("Missing per categorical:", cat_missing.sum().to_dict())
print("Rows with ALL of payment/salesperson/transport/warehouse missing:",
      cat_missing[["payment_type", "salesperson", "transport_method", "warehouse"]].all(axis=1).sum())
print("Same in test:", test[["payment_type", "salesperson", "transport_method", "warehouse"]].isna().all(axis=1).sum())

Missing per categorical: {'payment_type': 195, 'salesperson': 194, 'commercial_zone': 0, 'transport_method': 196, 'warehouse': 195, 'document_series': 0}
Rows with ALL of payment/salesperson/transport/warehouse missing: 0
Same in test: 0


### Key insight - Section 1

**Found** - most missing values are missing *by definition*:
1. **1st purchase (603 rows):** no history, so recency, mean/min/max and gaps don't exist.
2. **2nd purchase (445 rows):** one past purchase is not enough for a standard deviation or for a gap between past purchases (`prior_std_*`, `prior_mean_gap_days`).
3. **3rd purchase (365 rows):** two past purchases give only one gap, so its standard deviation (`prior_std_gap_days`) doesn't exist.
4. **Random:** about 195 values are missing for no reason in `days_since_previous_purchase`, `prior_mean_gap_days` and `spend_365d` (the same number as in the categoricals). The rest of the `unexplained` column are the values set to NaN in notebook 01 (Section 3).

- Of the 607 customers in train, 603 have their 1st purchase in the data and 445 came back at least once (about 74%).
- `payment_type`, `salesperson`, `transport_method` and `warehouse` have about 195 missing values each (30 in test), almost never on the same row.

**For preprocessing**
- Missing *by definition* means "no history", not "unknown value". Add an `is_first_purchase` flag (and use `prior_purchase_count` to mark 2nd/3rd purchases), then fill these NaN with a constant. The flag tells the model what the constant means.
- Random NaN: impute with the median (numeric) or a "missing" category (categorical), fitted on train only.
- Tree models such as HistGradientBoosting can also handle NaN directly; compare both options in validation.

## 2. The two targets

In [44]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("repeat_purchase_90d", "spend_next_90d (EUR)",
                                                    "log(1 + spend_next_90d)"))
counts = df[TARGET].value_counts().sort_index()
fig.add_bar(x=["0 - no repeat", "1 - repeat"], y=counts.values, marker_color=[GREY, BLUE],
            text=[f"{v} ({v / len(df):.0%})" for v in counts.values], textposition="outside", row=1, col=1)

returned = df[df[TARGET] == 1]
fig.add_histogram(x=returned[SPEND].dropna(), nbinsx=60, marker_color=BLUE, row=1, col=2)
fig.add_histogram(x=np.log1p(returned[SPEND].dropna()), nbinsx=60, marker_color=BLUE, row=1, col=3)
fig.update_xaxes(title_text="EUR", row=1, col=2)
fig.update_xaxes(title_text="log EUR", row=1, col=3)
fig.update_yaxes(range=[0, counts.max() * 1.15], row=1, col=1)
fig.update_layout(showlegend=False, height=420)
fig.show()

print("Skewness raw:", round(returned[SPEND].skew(), 2), "| log:", round(np.log1p(returned[SPEND]).skew(), 2))
print("Spend present when target = 0:", df.loc[df[TARGET] == 0, SPEND].notna().sum())
print("Spend missing when target = 1:", returned[SPEND].isna().sum())

Skewness raw: 3.04 | log: -0.56
Spend present when target = 0: 0
Spend missing when target = 1: 20


The 196 returners without a spend value were handled in notebook 01 (Section 5): 176 were recovered, so only 20 are still missing.

### Key insight - Section 2

**Found**
- **Classification target:** 56% repeat, 44% don't. Always answering "yes" already gives 56% accuracy.
- **Regression target:** very skewed. After `log1p` it is almost symmetric.

**For preprocessing**
- Classes are balanced enough: no resampling or class weights needed. Compare models with ROC-AUC or F1 (check which metric Kaggle uses), not accuracy.
- Train the regression on `log1p(spend)` and convert predictions back with `expm1`.

## 3. Time

In [45]:
quarterly_train = df.set_index("purchase_date").resample("QE").size()
quarterly_test = test.set_index("purchase_date").resample("QE").size()
rate = df.set_index("purchase_date").resample("QE")[TARGET].mean()

# Two panels with a shared time axis (instead of two y-axes on one chart)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4], vertical_spacing=0.1,
                    subplot_titles=("Purchases per quarter", "Repeat rate per quarter (train)"))
bar_width = 80 * 24 * 3600 * 1000  # 80 days, in milliseconds
fig.add_bar(x=quarterly_train.index, y=quarterly_train.values, width=bar_width, marker_color=BLUE, name="train", row=1, col=1)
fig.add_bar(x=quarterly_test.index, y=quarterly_test.values, width=bar_width, marker_color=ORANGE, name="test", row=1, col=1)
fig.add_scatter(x=rate.index, y=rate.values, mode="lines+markers", line=dict(color=BLUE, width=2),
                marker=dict(size=8), name="repeat rate", showlegend=False, row=2, col=1)
fig.update_yaxes(title_text="purchases", row=1, col=1)
fig.update_yaxes(title_text="repeat rate", range=[0, 1], row=2, col=1)
fig.update_layout(title="Purchases over time: test comes strictly after train", height=600)
fig.show()

print("train:", df["purchase_date"].min().date(), "->", df["purchase_date"].max().date())
print("test: ", test["purchase_date"].min().date(), "->", test["purchase_date"].max().date())

train: 2015-07-11 -> 2025-02-15
test:  2025-05-24 -> 2026-10-06


In [46]:
yearly = df.groupby(df["purchase_date"].dt.year)[TARGET].agg(["mean", "size"])
monthly = df.groupby(df["purchase_date"].dt.month)[TARGET].mean()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Repeat rate per year", "Repeat rate by month of the current purchase"))
fig.add_scatter(x=yearly.index, y=yearly["mean"], mode="lines+markers", line=dict(color=BLUE, width=2),
                marker=dict(size=8), customdata=yearly["size"],
                hovertemplate="%{x}: %{y:.2f} (n=%{customdata})<extra></extra>", row=1, col=1)
fig.add_bar(x=monthly.index, y=monthly.values, marker_color=BLUE, row=1, col=2)
fig.add_hline(y=df[TARGET].mean(), line_dash="dash", line_color=ORANGE, row=1, col=2)
fig.update_yaxes(title_text="repeat rate", range=[0.3, 0.8], row=1, col=1)
fig.update_yaxes(range=[0, 1], row=1, col=2)
fig.update_xaxes(dtick=1, row=1, col=1)
fig.update_xaxes(tickmode="linear", tick0=1, dtick=1, row=1, col=2)
fig.update_layout(showlegend=False, height=420)
fig.show()

### Key insight - Section 3

**Found**
- Train covers Jul 2015 - Feb 2025 and test covers May 2025 - Oct 2026. No overlap: we are predicting the future.
- The repeat rate per year moves between 0.48 and 0.63, with no trend.
- There is a month effect: March and November about 0.45, January, May and September about 0.70.

**For preprocessing and validation**
- Use **time-based validation**: train on older data and validate on the most recent 15-20% (or `TimeSeriesSplit`). Random K-fold would give scores that are too optimistic.
- Test `month` as a feature (one-hot, or sine/cosine). Dates are anonymised, so keep it only if it helps in validation.
- Don't use the date itself as a number: test dates are outside the train range.

## 4. What drives a repeat purchase?

In [47]:
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.18, horizontal_spacing=0.08,
                    subplot_titles=("Recency: days since the previous purchase", "Frequency: number of prior purchases",
                                    "Purchases in the previous 90 days", "Value of the current purchase"))
rate_by_bins(df, "days_since_previous_purchase", [0, 7, 14, 30, 60, 90, 180, 365, 5000], fig=fig, row=1, col=1,
             xlabel="days")
rate_by_bins(df, "prior_purchase_count", [-1, 0, 1, 2, 5, 10, 25, 50, 100, 300], fig=fig, row=1, col=2,
             labels=["0", "1", "2", "3-5", "6-10", "11-25", "26-50", "51-100", "100+"], xlabel="prior purchases")
rate_by_bins(df, "purchase_count_90d", [-1, 0, 1, 2, 3, 5, 10, 30], fig=fig, row=2, col=1,
             labels=["0", "1", "2", "3", "4-5", "6-10", "10+"], xlabel="purchases")
rate_by_bins(df, "Total a Pagar", [0, 50, 150, 400, 1000, 3000, 30000], fig=fig, row=2, col=2, xlabel="EUR")
fig.update_layout(title=f"Repeat rate by recency, frequency and value<br><sup>dashed line = overall rate ({df[TARGET].mean():.2f})</sup>",
                  height=900, margin_t=120)
fig.show()

- **Frequency** is the strongest signal: 1 prior purchase -> 37% repeat, 100+ -> 99%.
- **Recency** breaks at 90 days: last purchase within 90 days -> about 70% repeat; longer ago -> 30-40%.
- **Purchases in the last 90 days:** steady rise from 38% (none) to 100% (10+).
- **Value of the current purchase:** small effect (between 52% and 67%).

In [48]:
# Does the customer's usual rhythm predict the outcome? Compare the typical gap with the 90-day window
fig = make_subplots(rows=1, cols=2, subplot_titles=("Average gap between past purchases", "Purchases in the previous 365 days"))
for value, color, label in [(0, GREY, "no repeat"), (1, BLUE, "repeat")]:
    subset = df[df[TARGET] == value]
    add_density(fig, np.log1p(subset["prior_mean_gap_days"]), color, label, row=1, col=1)
    add_density(fig, np.log1p(subset["purchase_count_365d"]), color, label, row=1, col=2, showlegend=False)
fig.add_vline(x=np.log1p(90), line_dash="dash", line_color=RED, annotation_text="90 days", row=1, col=1)
fig.update_xaxes(title_text="log(1 + days)", row=1, col=1)
fig.update_xaxes(title_text="log(1 + count)", row=1, col=2)
fig.update_yaxes(title_text="density", row=1, col=1)
fig.update_layout(height=450)
fig.show()

df["gap_below_90"] = df["prior_mean_gap_days"] < 90
print(df.groupby("gap_below_90")[TARGET].agg(["mean", "size"]).round(3))
df = df.drop(columns="gap_below_90")

               mean  size
gap_below_90             
False         0.376  3164
True          0.735  3305


In [49]:
first = df["prior_purchase_count"] == 0
print("Repeat rate, first purchase:     ", round(df.loc[first, TARGET].mean(), 3), f"(n={first.sum()})")
print("Repeat rate, returning customers:", round(df.loc[~first, TARGET].mean(), 3), f"(n={(~first).sum()})")

Repeat rate, first purchase:      0.333 (n=603)
Repeat rate, returning customers: 0.583 (n=5866)


### Key insight - Section 4

**Found**
- **Frequency** is the strongest signal: the more past purchases (in total and in the last 90 days), the more likely a repeat.
- **Recency:** a last purchase within 90 days repeats about 70%; longer ago, 30-40%.
- 1st purchases repeat 33%, returning customers 58%.
- Customers whose average gap is under 90 days repeat 74%, the others 38%.
- The value of the current purchase has only a small effect.

**For preprocessing**
- These patterns are turned into features in notebook 03 (`is_first_purchase`, "is the customer late compared with their usual rhythm?").
- Which columns predict the target best, and which are near copies of each other, is also checked there.

## 5. What drives how much they spend?

In [50]:
returned = df[(df[TARGET] == 1) & df[SPEND].notna()].copy()
returned["log_spend"] = np.log1p(returned[SPEND])

spend_cols = ["spend_365d", "prior_mean_purchase_value", "Total a Pagar"]
rhos = [spearmanr(returned[col], returned[SPEND], nan_policy="omit")[0] for col in spend_cols]
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{col}  (rho = {rho:.2f})" for col, rho in zip(spend_cols, rhos)])
for i, col in enumerate(spend_cols, start=1):
    fig.add_scatter(x=np.log1p(returned[col]), y=returned["log_spend"], mode="markers",
                    marker=dict(size=4, color=BLUE, opacity=0.25), showlegend=False, row=1, col=i)
    fig.update_xaxes(title_text=f"log(1 + {col})", row=1, col=i)
fig.update_yaxes(title_text="log(1 + spend_next_90d)", row=1, col=1)
fig.update_layout(height=480)
fig.show()

In [51]:
labels = ["0", "1", "2", "3", "4-5", "6-10", "10+"]
groups = pd.cut(returned["purchase_count_90d"], [-1, 0, 1, 2, 3, 5, 10, 30], labels=labels)
box_data = returned.assign(group=groups).dropna(subset=["group"])
fig = px.box(box_data, x="group", y=SPEND, points=False, log_y=True,
             category_orders={"group": labels}, color_discrete_sequence=[BLUE])
fig.update_layout(title="Frequent buyers return AND spend more",
                  xaxis_title="purchases in the previous 90 days", yaxis_title="spend_next_90d (EUR, log)")
fig.show()

### Key insight - Section 5

**Found**
- How much a customer spends next depends mostly on **past spend** (`spend_365d`, correlation 0.60).
- The value of the current purchase matters much less.
- Frequent buyers return **and** spend more.
- Spend ranges from about 1 EUR to 25,600 EUR.

**For preprocessing**
- Regression on `log1p(spend)`; also test `log1p` on the spend features for linear models.
- On the raw scale, a few big customers dominate errors such as RMSE. Evaluate with the metric Kaggle uses.
- Which features to use for the spend model is checked in notebook 03.

## 6. Categorical variables

In [52]:
# Overview of each categorical column: how many levels, how concentrated, how many are rare or new in test
overview = pd.DataFrame({
    "unique levels (train)": df[cat_cols].nunique(),
    "unique levels (test)": test[cat_cols].nunique(),
    "levels only in test": [len(set(test[c].dropna()) - set(df[c].dropna())) for c in cat_cols],
    "levels with < 20 rows": [(df[c].value_counts() < 20).sum() for c in cat_cols],
    "top level share": [df[c].value_counts(normalize=True).iloc[0] for c in cat_cols],
    "missing": df[cat_cols].isna().sum(),
})
overview["top level share"] = (overview["top level share"] * 100).round(1).astype(str) + "%"
overview

,unique levels (train),unique levels (test),levels only in test,levels with < 20 rows,top level share,missing
payment_type,9,11,3,3,72.8%,195
salesperson,9,3,0,3,64.6%,194
commercial_zone,5,5,2,2,96.8%,0
transport_method,80,29,2,46,17.8%,196
warehouse,2,2,1,0,99.7%,195
document_series,3,1,0,0,42.3%,0


In [53]:
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.15,
                    subplot_titles=[f"{col} ({df[col].nunique()} levels)" for col in cat_cols])
for i, col in enumerate(cat_cols):
    filled = df[col].fillna("MISSING")
    summary = filled.value_counts().to_frame("n")
    summary["rate"] = df.groupby(filled)[TARGET].mean()
    summary = summary.head(10)
    labels = [s if s != "MISSING" else "NA" for s in summary.index]
    position = dict(row=i // 3 + 1, col=i % 3 + 1)
    fig.add_bar(x=labels, y=summary["rate"], marker_color=[RED if n < 30 else BLUE for n in summary["n"]],
                text=summary["n"], textposition="outside", showlegend=False,
                hovertemplate="%{x}<br>repeat rate: %{y:.2f}<br>rows: %{text}<extra></extra>", **position)
    fig.add_hline(y=df[TARGET].mean(), line_dash="dash", line_color=ORANGE, **position)
    fig.update_yaxes(range=[0, 1.15], **position)
fig.update_yaxes(title_text="repeat rate", col=1)
fig.update_layout(title="Repeat rate per level (top 10 by size; red = fewer than 30 rows; number = rows)", height=750)
fig.show()

In [54]:
print("Levels in test that never appear in train:")
for col in cat_cols:
    unseen = set(test[col].dropna()) - set(df[col].dropna())
    if unseen:
        print(f"  {col}: {sorted(unseen)}")

Levels in test that never appear in train:
  payment_type: ['1', '5', '6']
  commercial_zone: ['1', '5']
  transport_method: ['54', '65']
  warehouse: ['2']


### Key insight - Section 6

**Found**
- `payment_type` 9: 95% repeat (502 rows), against 53% for the main type 7. Maybe a recurring-payment agreement.
- `salesperson`: salesperson 1 repeats 62%, salesperson 4 39%.
- `transport_method`: 80 levels, 46 with fewer than 20 rows. Some large levels are very informative (level 11: 98% of 253 rows).
- `document_series`: no effect (56% in all three).
- `commercial_zone` (97% in one level) and `warehouse` (99.7% in one level) are almost constant.
- test has levels never seen in train in 4 columns.

**For preprocessing**
- Drop `document_series`. `commercial_zone` and `warehouse` carry little information: drop them or keep only a flag for the rare levels.
- Group rare levels (fewer than 20 rows) into "other", especially in `transport_method`.
- Encode with one-hot for the small columns. For `transport_method`, test frequency or target encoding (target encoding fitted inside each fold, to avoid leakage).
- The encoder must accept unknown levels (`OneHotEncoder(handle_unknown="ignore")`), or the pipeline will fail on test.
- Missing values become their own "missing" category.

## 7. Customers and `clientes.csv`

In [55]:
rows_per_customer = df["customer_id"].value_counts()
share = rows_per_customer.sort_values(ascending=False).cumsum() / len(df)
customer_rate = df.groupby("customer_id")[TARGET].mean()

fig = make_subplots(rows=1, cols=3, subplot_titles=("Purchases per customer in train", "Concentration of rows",
                                                    "Repeat rate per customer"))
fig.add_histogram(x=np.log10(rows_per_customer), nbinsx=30, marker_color=BLUE, row=1, col=1)
fig.add_scatter(x=np.arange(1, len(share) + 1) / len(share), y=share.values, mode="lines",
                line=dict(color=BLUE, width=2), row=1, col=2)
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=GREY, dash="dash"), row=1, col=2)
fig.add_histogram(x=customer_rate, nbinsx=20, marker_color=BLUE, row=1, col=3)

ticks = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
fig.update_xaxes(title_text="rows (log scale)", tickvals=np.log10(ticks), ticktext=ticks, row=1, col=1)
fig.update_xaxes(title_text="share of customers (largest first)", row=1, col=2)
fig.update_xaxes(title_text="share of purchases followed by another", row=1, col=3)
fig.update_yaxes(title_text="customers", row=1, col=1)
fig.update_yaxes(title_text="share of rows", row=1, col=2)
fig.update_yaxes(title_text="customers", row=1, col=3)
fig.update_layout(showlegend=False, height=450)
fig.show()

print("Customers:", len(rows_per_customer), "| with 1 row:", (rows_per_customer == 1).sum(),
      "| top 10% of customers hold", f"{share.iloc[int(len(share) * 0.1)]:.0%}", "of rows")
print("Customers who NEVER repeat within 90 days:", (customer_rate == 0).sum())
print("test rows whose customer is in train:", f"{test['customer_id'].isin(df['customer_id']).mean():.1%}")

Customers: 607 | with 1 row: 157 | top 10% of customers hold 50% of rows
Customers who NEVER repeat within 90 days: 256
test rows whose customer is in train: 94.5%


In [56]:
merged = df.merge(clientes, on="customer_id", how="left")
print("train rows with a match in clientes:", merged["country_or_region"].notna().mean())

top_countries = merged["country_or_region"].value_counts().head(4).index
sub = merged[merged["country_or_region"].isin(top_countries)]
country_rate = sub.groupby("country_or_region")[TARGET].agg(["mean", "size"])
island_rate = merged.groupby(merged["island"].isna())[TARGET].agg(["mean", "size"])

fig = make_subplots(rows=1, cols=2, subplot_titles=("Repeat rate by country", "Repeat rate by whether 'island' is filled"))
fig.add_bar(x=[c.split(":")[0].replace("Country ", "") for c in country_rate.index], y=country_rate["mean"],
            marker_color=BLUE, text=[f"n={n}" for n in country_rate["size"]], textposition="outside", row=1, col=1)
fig.add_bar(x=["island known", "island missing"], y=island_rate["mean"], marker_color=[BLUE, ORANGE],
            text=[f"n={n}" for n in island_rate["size"]], textposition="outside", row=1, col=2)
fig.update_yaxes(range=[0, 1], row=1)
fig.update_yaxes(title_text="repeat rate", row=1, col=1)
fig.update_layout(showlegend=False, height=420)
fig.show()

first_year = df.groupby("customer_id")["purchase_date"].min().dt.year
since = clientes.set_index("customer_id")["customer_since_year"].reindex(first_year.index)
print("customer_since_year vs first purchase year in train:",
      {"earlier": (since < first_year).sum(), "same": (since == first_year).sum(), "later": (since > first_year).sum()})

train rows with a match in clientes: 1.0


customer_since_year vs first purchase year in train: {'earlier': np.int64(197), 'same': np.int64(410), 'later': np.int64(0)}


### Key insight - Section 7

**Found**
- A few customers dominate: median 4 rows per customer, maximum 276, and the top 10% of customers hold 50% of the rows. 157 customers appear only once.
- 256 of the 607 customers never repeat within 90 days.
- 94.5% of the test rows belong to customers already in train.
- In `clientes.csv`, two things are related to the target: the **country** (CT5 repeats 39%, CT10 57%) and whether **`island` is filled** (64% vs 52%). The other columns are anonymised addresses with too many levels. `customer_since_year` agrees with the purchase history.

**For preprocessing**
- Join `clientes.csv` on `customer_id` and add `country` (small countries grouped into "other") and `island_missing`.
- Country CT5 overlaps almost completely with 0% VAT (notebook 03): keep one of the two.
- Drop the other address columns and `customer_type`.

## 8. Is test similar to train?

In [57]:
compare_cols = ["customer_tenure_days", "prior_purchase_count", "days_since_previous_purchase", "Total a Pagar"]
fig = make_subplots(rows=1, cols=4, subplot_titles=compare_cols)
for i, col in enumerate(compare_cols, start=1):
    for data, color, label in [(df, BLUE, "train"), (test, ORANGE, "test")]:
        values = np.sort(np.log1p(data[col].dropna().clip(lower=0)))
        fig.add_scatter(x=values, y=np.arange(1, len(values) + 1) / len(values), mode="lines",
                        line=dict(color=color, width=2), name=label, legendgroup=label, showlegend=(i == 1),
                        row=1, col=i)
    fig.update_xaxes(title_text="log(1 + value)", row=1, col=i)
fig.update_yaxes(title_text="cumulative share", row=1, col=1)
fig.update_layout(title="Distribution of key features: train vs test", height=430)
fig.show()

print("First purchases - train:", f"{(df['prior_purchase_count'] == 0).mean():.1%}", "| test:", f"{(test['prior_purchase_count'] == 0).mean():.1%}")
pd.DataFrame({"train median": df[compare_cols].median(), "test median": test[compare_cols].median()}).round(1)

First purchases - train: 9.3% | test: 2.2%


,train median,test median
customer_tenure_days,918.0,2662.0
prior_purchase_count,11.0,28.0
days_since_previous_purchase,57.0,60.5
Total a Pagar,476.7,663.9


### Key insight - Section 8

**Found**
- Test customers are older and more loyal: median tenure 2,662 vs 918 days, 28 vs 11 prior purchases, and only 2.2% first purchases vs 9.3%. Recency is similar.
- So test will probably have a higher repeat rate than train.

**For preprocessing and validation**
- Validate on the most recent part of train: it is the most similar to test.
- Check the decision threshold on that recent validation period, not on the whole of train.

## Summary: what preprocessing must do

| # | Finding | Decision |
|---|---|---|
| 1 | NaN because there is no history (1st / 2nd / 3rd purchase: 603 / 445 / 365 rows) | flag + constant |
| 1 | Random NaN (about 195 per column) | impute (median or "missing" category), fitted on train only |
| 2 | Classes 56 / 44; spend very skewed | no resampling; regression on `log1p(spend)` |
| 3 | Test comes strictly after train | time-based validation |
| 6 | Rare levels; levels seen only in test | group rare levels, accept unknown levels |
| 7 | `clientes.csv`: only country and `island` are useful | join, keep `country` and `island_missing` |
| 8 | Test customers are older and more loyal | validate on the most recent part of train |

**Next**
- Imputation, encoding and scaling inside the sklearn `Pipeline`, fitted on train only.
- New features and feature selection: `03_feature_engineering_selection.ipynb`.